# Clase 06 — Análisis de sentimiento clásico

**Curso:** NLP Clásico con Python · **Bloque 2:** Representar y clasificar
**Tiempo estimado:** 100 minutos · **Capítulo del manual:** `manual/06_sentimiento_clasico.md`

## Qué vas a ver en esta clase

- Un **analizador léxico** de sentimiento: un pequeño **léxico de polaridad** (lista de palabras con un valor de −2 a +2) y una suma de puntuaciones.
- **Negación** (una ventana de palabras) e **intensificadores** (`muy`, `poco`): qué arreglan y dónde se quedan cortos.
- Un clasificador **supervisado** (TF-IDF + regresión logística) y su **comparación con el léxico sobre el mismo corpus y la misma partición**.
- Por qué la **bolsa de palabras pierde la negación**, con un experimento guiado.
- El **sarcasmo y la ironía** como límite, y el **análisis de errores por categorías** (en lugar de un único porcentaje).

## Requisitos

- Bolsa de palabras y TF-IDF (clase 4) y clasificación con regresión logística, partición entrenamiento/prueba y métricas (clase 5).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scikit-learn`. La celda siguiente instala solo lo que falte (funciona igual en Google Colab).
- La carpeta `datos/` del curso: `resenas_es.csv`, `resenas_it.csv` y los dos léxicos `lexico_polaridad_es.csv`, `lexico_polaridad_it.csv`. En Colab, súbela desde el panel de archivos.

> **Advertencia sobre las cifras:** el corpus es sintético y pequeño (300 reseñas por idioma) y el léxico es un **léxico minúsculo escrito para el curso**. Todas las cifras son un *ejemplo ilustrativo*: no dicen que un método sea mejor que otro en general, y **no prometemos que el léxico gane al modelo supervisado: medimos lo que sale**.

> **Cómo usar este notebook:** ejecuta las celdas en orden. Es una clase densa (100 minutos): las secciones 6 y 8 se pueden dejar como lectura si falta tiempo.

---
## Instalación

Instala solo lo que falte. En un entorno con todo instalado no hace nada.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("scipy", "scipy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, fijamos la **semilla**, definimos `cargar_datos` y leemos las reseñas. Hacemos **una sola partición** entrenamiento/prueba (70/30, estratificada por etiqueta) y la usaremos para todo: el léxico y el modelo supervisado se evalúan sobre **las mismas 90 reseñas de prueba** de cada idioma.

In [ ]:
import os
import re
import random
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

random.seed(42)                          # semilla fija: todos obtenemos los mismos resultados
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

IDIOMAS = ("spa", "ita")
ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
ARCHIVO_LEXICO = {"spa": "lexico_polaridad_es.csv", "ita": "lexico_polaridad_it.csv"}
CLASES = ["negativo", "neutro", "positivo"]

resenas = {i: pd.read_csv(cargar_datos(a)) for i, a in ARCHIVO_RESENAS.items()}

# Una única partición entrenamiento/prueba para TODO el notebook (la misma para el léxico y para el modelo)
train, test = {}, {}
for i, df in resenas.items():
    train[i], test[i] = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    print(i, "total", len(df), "· train", len(train[i]), "· test", len(test[i]))
print("Datos y librerías cargados correctamente.")

---
## 1. El corpus: reseñas con tres clases y casos difíciles

Las reseñas están etiquetadas `positivo`, `negativo` o `neutro`, con las clases equilibradas. La columna `dificil` marca (≈ 15 %) los casos con negación, ironía o juicios mixtos que **escribimos a propósito** para poner a prueba los métodos. Miramos solo el **entrenamiento**: la prueba se reserva para medir.

In [ ]:
for idioma in IDIOMAS:
    df = train[idioma]
    print(f"=== {idioma} ===")
    print(pd.crosstab(df["etiqueta"], df["dificil"], margins=True))
    print(df.sort_values("id").groupby("etiqueta").head(1)[["id", "etiqueta", "dificil", "texto_limpio"]].to_string(index=False))
    print()

**¿Qué observas?** Las tres clases están equilibradas y los casos `dificil=1` se reparten entre ellas. Fíjate en que la clase `neutro` **no es la ausencia de opinión**: incluye reseñas con elogios y quejas a la vez («bonito pero con pocas opciones»). Esto va a pesar mucho más adelante.

---
## 2. El léxico de polaridad: qué contiene y cómo se compiló

`datos/lexico_polaridad_es.csv` y `datos/lexico_polaridad_it.csv` son **listas escritas para el curso** (≈ 160 entradas por idioma, con valores −2, −1, +1, +2). Se compilaron **de memoria, a partir de vocabulario general de opinión, antes de medir nada sobre el corpus**, y no se han copiado de ningún léxico existente ni se han ajustado después mirando los errores. Solo contienen palabras con carga; los negadores y los intensificadores están en el código.

Es transparente (puedes leerlo entero) y no necesita datos etiquetados, pero es **minúsculo**: un léxico serio tiene miles de entradas.

In [ ]:
# datos-es
NEGADORES = {"spa": {"no", "ni", "nunca", "jamas", "tampoco", "sin", "nada", "ningun", "ninguno", "ninguna"},
             "ita": {"non", "mai", "senza", "nessun", "nessuno", "nessuna", "nemmeno", "neppure", "niente", "nulla"}}
INTENSIFICADORES = {"spa": {"muy": 1.5, "super": 1.5, "tremendamente": 1.5, "realmente": 1.5, "totalmente": 1.5,
                            "bastante": 1.25, "demasiado": 1.25, "poco": 0.5, "algo": 0.5},
                    "ita": {"molto": 1.5, "super": 1.5, "davvero": 1.5, "veramente": 1.5, "assolutamente": 1.5,
                            "estremamente": 1.5, "troppo": 1.25, "abbastanza": 1.25, "poco": 0.5, "alquanto": 0.75}}
CORTES = {"spa": {"pero", "aunque", "sino"}, "ita": {"ma", "pero", "sebbene", "tuttavia"}}   # las adversativas cortan la ventana

In [ ]:
def normalizar(texto):
    """Minúsculas y sin acentos: así el léxico también funciona con el texto bruto que perdió los acentos."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def tokenizar(texto):
    """Palabras y signos de puntuación (los signos delimitan la ventana de la negación)."""
    return re.findall(r"\w+|[.,;:!?()]", normalizar(texto))

def cargar_lexico(idioma):
    tabla = pd.read_csv(cargar_datos(ARCHIVO_LEXICO[idioma]))
    return {normalizar(p): float(v) for p, v in zip(tabla["palabra"], tabla["polaridad"])}

def analizar(texto, lexico, idioma, ventana=3, negacion=True, intensificadores=True):
    """Devuelve (puntuación total, lista de aportes (palabra, valor base, valor final))."""
    tokens = tokenizar(texto)
    total, aportes = 0.0, []
    for i, palabra in enumerate(tokens):
        if palabra not in lexico:
            continue
        valor = lexico[palabra]
        multiplicador, invertir = 1.0, False
        for j in range(i - 1, max(i - ventana, 0) - 1, -1):    # hacia atrás, hasta `ventana` tokens
            anterior = tokens[j]
            if not anterior[0].isalnum() or anterior in CORTES[idioma]:
                break                                          # puntuación o «pero»: fin de la cláusula
            if negacion and anterior in NEGADORES[idioma]:
                invertir = True                                # varios negadores seguidos invierten UNA sola vez
            elif intensificadores and anterior in INTENSIFICADORES[idioma]:
                multiplicador *= INTENSIFICADORES[idioma][anterior]
        final = valor * multiplicador * (-1 if invertir else 1)
        aportes.append((palabra, valor, final))
        total += final
    return total, aportes

def puntuar(texto, lexico, idioma, **opciones):
    return analizar(texto, lexico, idioma, **opciones)[0]

LEXICO = {idioma: cargar_lexico(idioma) for idioma in IDIOMAS}

In [ ]:
for idioma in IDIOMAS:
    tabla = pd.read_csv(cargar_datos(ARCHIVO_LEXICO[idioma]))
    print(f"=== {idioma}: {len(tabla)} entradas ===")
    print(tabla["polaridad"].value_counts().sort_index().to_dict())
    print(tabla.groupby("polaridad")["palabra"].apply(lambda s: ", ".join(list(s)[:6])).to_string())
    print()

### ¿El léxico «conoce» el corpus? Una comprobación de sospecha

Si el léxico se hubiera armado **copiando palabras del corpus** (y sobre todo de la prueba), casi todas sus entradas aparecerían en las reseñas y el acierto en prueba sería sospechosamente alto. Comprobamos tres cosas: cuántas entradas **no aparecen en ninguna reseña**, cuántas aparecen **solo en la prueba**, y si el acierto en entrenamiento y en prueba son parecidos (si estuviera afinado sobre la prueba, ahí sería mejor).

In [ ]:
def vocabulario(df, columna="texto_limpio"):
    return {t for texto in df[columna] for t in re.findall(r"\w+", normalizar(texto))}

for idioma in IDIOMAS:
    lex = LEXICO[idioma]
    v_train, v_test = vocabulario(train[idioma]), vocabulario(test[idioma])
    en_train = [p for p in lex if p in v_train]
    solo_test = [p for p in lex if p in v_test and p not in v_train]
    en_ninguna = [p for p in lex if p not in v_train and p not in v_test]
    print(f"{idioma}: {len(lex)} entradas · en el entrenamiento: {len(en_train)} · solo en la prueba: {len(solo_test)} · en ninguna reseña: {len(en_ninguna)}")
    for nombre, df in (("train", train[idioma]), ("test", test[idioma])):
        con_palabra = np.mean([bool(analizar(t, lex, idioma)[1]) for t in df["texto"]])
        print(f"   {nombre}: reseñas con al menos una palabra del léxico = {con_palabra:.3f}")

**¿Qué observas?** Una parte grande del léxico (más de un tercio en español y más de la mitad en italiano) **no aparece en ninguna reseña**: no se construyó a partir del corpus. Las entradas que aparecen solo en la prueba son ≈ 10 por idioma, y la **cobertura** (reseñas con al menos una palabra del léxico) se mueve entre el 52 % y el 74 % según el idioma y la partición, **sin una dirección común**: mayor en la prueba en español, menor en italiano (con 90 reseñas, esas diferencias son ruido de muestreo). Esto no *demuestra* que no haya sesgo, pero es coherente con un léxico general; el acierto en entrenamiento y en prueba se compara en la sección 4.

---
## 3. El analizador paso a paso

`analizar` recorre los tokens; para cada palabra del léxico mira **hacia atrás hasta `ventana` tokens** dentro de la misma cláusula (la puntuación y `pero`/`ma` la cortan):

- si encuentra un **negador** (`no`, `nunca`, `sin`, `ni`…), **invierte** el signo (una sola vez aunque haya varios: «no está nada mal» es positivo);
- si encuentra un **intensificador**, multiplica: `muy` ×1,5 · `poco` ×0,5.

Lo probamos con frases pequeñas cuya puntuación se calcula **a mano** (con `ventana=3`).

In [ ]:
# datos-es
FRASES = {
    "spa": ["el servicio es bueno",
            "el servicio no es bueno",
            "el servicio es muy bueno",
            "no es nada malo",
            "ni bueno ni malo",
            "bueno pero caro",
            "sin problemas",
            "no llegó tarde"],
    "ita": ["il servizio è buono",
            "il servizio non è buono",
            "il servizio è molto buono",
            "non è per niente male",
            "né buono né cattivo",
            "buono ma caro",
            "senza problemi",
            "non è arrivato in ritardo"],
}

In [ ]:
ESPERADAS = [1.0, -1.0, 1.5, 1.0, 0.0, 0.0, 1.0, 1.0]     # calculadas a mano
ESPERADAS_ITA = [1.0, -1.0, 1.5, 1.0, 0.0, 0.0, 1.0, -1.0]  # «non è arrivato in ritardo»: el negador queda a 4 tokens

for idioma, esperadas in (("spa", ESPERADAS), ("ita", ESPERADAS_ITA)):
    filas = []
    for frase, esperada in zip(FRASES[idioma], esperadas):
        con = puntuar(frase, LEXICO[idioma], idioma)
        sin = puntuar(frase, LEXICO[idioma], idioma, negacion=False)
        assert con == esperada, (frase, con, esperada)
        filas.append({"frase": frase, "sin negación": sin, "con negación": con})
    print(pd.DataFrame(filas).to_string(index=False))
    print()
print("Todas las puntuaciones coinciden con el cálculo a mano ✔")

**¿Qué observas?** Sin negación, «el servicio no es bueno» puntúa **+1** (el mismo signo que «es bueno»); con la ventana pasa a **−1**. «Ni bueno ni malo» da 0 (`ni` invierte los dos) y «bueno pero caro» da 0 (`pero` corta la ventana). Y hay un fallo instructivo: en italiano, «non è arrivato in ritardo» queda en **−1** porque `non` está a **cuatro** tokens de `ritardo` y la ventana es 3. Con `ventana=4` se arreglaría, pero una ventana más ancha empieza a invertir palabras que nada tienen que ver: **no hay valor perfecto**.

### Transparencia: por qué una reseña recibe su puntuación

Una ventaja real del léxico: se puede **explicar** cada decisión palabra por palabra.

In [ ]:
for idioma in IDIOMAS:
    candidatas = train[idioma][train[idioma]["dificil"] == 0].sort_values("id")
    fila = next(f for _, f in candidatas.iterrows() if len(analizar(f["texto_limpio"], LEXICO[idioma], idioma)[1]) >= 3)
    total, aportes = analizar(fila["texto_limpio"], LEXICO[idioma], idioma)
    print(f"[{idioma}] ({fila['etiqueta']}) {fila['texto_limpio']}")
    for palabra, base, final in aportes:
        print(f"   {palabra:<14} base {base:+.1f}  →  {final:+.2f}")
    print(f"   TOTAL = {total:+.2f}\n")

---
## 4. De la puntuación a la clase: umbral y ventana, elegidos con el entrenamiento

La puntuación es un número; para obtener `positivo` / `neutro` / `negativo` necesitamos un **umbral**: por encima, positivo; por debajo de −umbral, negativo; en medio, neutro. Es el **único parámetro que ajustamos**, y lo hacemos **solo con el entrenamiento**. Aprovechamos para ver cuánto influye la ventana de la negación (sobre el texto bruto, igual que el modelo supervisado).

In [ ]:
def clasificar(puntuacion, umbral):
    """Puntuación → clase: por encima del umbral, positivo; por debajo de −umbral, negativo; en medio, neutro."""
    if puntuacion > umbral:
        return "positivo"
    if puntuacion < -umbral:
        return "negativo"
    return "neutro"

REJILLA_UMBRAL = (0, 0.25, 0.5, 0.75, 1.0)

def mejor_umbral(puntuaciones, y, rejilla=REJILLA_UMBRAL):
    """Umbral con mayor exactitud sobre (puntuaciones, y); a igualdad, el menor."""
    mejor = max(rejilla, key=lambda u: (accuracy_score(y, [clasificar(p, u) for p in puntuaciones]), -u))
    return mejor, accuracy_score(y, [clasificar(p, mejor) for p in puntuaciones])

def modelo_supervisado(ngramas=(1, 2)):
    return make_pipeline(TfidfVectorizer(ngram_range=ngramas, sublinear_tf=True),
                         LogisticRegression(max_iter=1000))

In [ ]:
CONFIGS = {
    "sin negación": dict(negacion=False),
    "ventana 1": dict(ventana=1),
    "ventana 2": dict(ventana=2),
    "ventana 3 (por defecto)": dict(ventana=3),
    "ventana 4": dict(ventana=4),
    "ventana 5": dict(ventana=5),
    "sin intensificadores": dict(intensificadores=False),
}
filas = []
for nombre, opciones in CONFIGS.items():
    fila = {"configuración": nombre}
    for idioma in IDIOMAS:
        puntos = [puntuar(t, LEXICO[idioma], idioma, **opciones) for t in train[idioma]["texto"]]
        umbral, acierto = mejor_umbral(puntos, train[idioma]["etiqueta"])
        fila[f"{idioma} umbral"] = umbral
        fila[f"{idioma} exactitud (train)"] = round(acierto, 3)
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** Las diferencias entre configuraciones son **pequeñas**: con 210 reseñas de entrenamiento, un punto de exactitud son ≈ 2 reseñas. No leas en la tabla una «ventana óptima»: es ruido dentro de lo que esta muestra permite distinguir. Mantenemos `ventana=3` por principio (la ventana por defecto del capítulo) y fijamos el **umbral con el entrenamiento**.

In [ ]:
UMBRAL = {}
for idioma in IDIOMAS:
    puntos = [puntuar(t, LEXICO[idioma], idioma) for t in train[idioma]["texto"]]
    UMBRAL[idioma], acierto_train = mejor_umbral(puntos, train[idioma]["etiqueta"])
    puntos_test = [puntuar(t, LEXICO[idioma], idioma) for t in test[idioma]["texto"]]
    acierto_test = accuracy_score(test[idioma]["etiqueta"], [clasificar(p, UMBRAL[idioma]) for p in puntos_test])
    print(f"{idioma}: umbral elegido = {UMBRAL[idioma]} · exactitud del léxico en train = {acierto_train:.3f} · en test = {acierto_test:.3f}")

Si el léxico estuviera afinado sobre la prueba, el acierto en prueba sería claramente **mayor** que en entrenamiento. No es lo que vemos (con 90 reseñas de prueba, unos pocos puntos de diferencia son ruido), lo que es coherente con la compilación descrita en la sección 2.

---
## 5. El clasificador supervisado y la comparación en la misma partición

Entrenamos `TfidfVectorizer` (unigramas y bigramas, `sublinear_tf`) + `LogisticRegression` **con el mismo entrenamiento** y lo medimos en **la misma prueba**, sobre el mismo texto bruto. Comparamos exactitud, F1 macro y la exactitud en los casos `dificil` (con muy pocas reseñas: se indica `n`).

In [ ]:
def resumen(y, pred, dificil):
    y, pred, dificil = np.asarray(y), np.asarray(pred), np.asarray(dificil)
    return {"exactitud": round(accuracy_score(y, pred), 3),
            "F1 macro": round(f1_score(y, pred, average="macro"), 3),
            "exactitud fácil": round(accuracy_score(y[dificil == 0], pred[dificil == 0]), 3),
            "exactitud difícil": round(accuracy_score(y[dificil == 1], pred[dificil == 1]), 3),
            "n difícil": int((dificil == 1).sum())}

PRUEBA = {}
filas = []
for idioma in IDIOMAS:
    y = test[idioma]["etiqueta"]
    pred_lex = [clasificar(puntuar(t, LEXICO[idioma], idioma), UMBRAL[idioma]) for t in test[idioma]["texto"]]
    modelo = modelo_supervisado().fit(train[idioma]["texto"], train[idioma]["etiqueta"])
    pred_lr = modelo.predict(test[idioma]["texto"])
    PRUEBA[idioma] = {"lex": pred_lex, "lr": pred_lr}
    for nombre, pred in (("léxico", pred_lex), ("TF-IDF + LR", pred_lr)):
        filas.append({"idioma": idioma, "método": nombre, **resumen(y, pred, test[idioma]["dificil"])})
TABLA = pd.DataFrame(filas)
print(TABLA.to_string(index=False))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for eje, idioma in zip(ejes, IDIOMAS):
    sub = TABLA[TABLA["idioma"] == idioma]
    eje.bar(sub["método"], sub["exactitud"], color=["#8fb4d9", "#e9a15b"])
    eje.axhline(1 / 3, color="gray", linestyle="--", linewidth=1)
    eje.set_title(f"{idioma} (n = {len(test[idioma])})")
    eje.set_ylim(0, 1)
ejes[0].set_ylabel("exactitud en prueba")
plt.suptitle("Misma partición y mismo texto (la línea gris es el azar: 1/3)")
plt.tight_layout()
plt.show()

**¿Qué observas?** Compara las dos filas de cada idioma. Con **90 reseñas de prueba**, una reseña vale ≈ 1,1 puntos de exactitud y las diferencias de pocos puntos **no son concluyentes**. Y en los casos `dificil` hay una decena de reseñas: ahí las cifras son aún más inestables. En la sección siguiente repetimos la partición muchas veces para no depender de una sola.

---
## 6. ¿Depende de la partición? Veinte particiones distintas

Repetimos 20 veces la partición 70/30 (semillas 0 a 19): en cada una el umbral del léxico se elige con **su** entrenamiento y el modelo supervisado se entrena con **su** entrenamiento. Es una comprobación de robustez, no una prueba estadística: las 20 pruebas se solapan.

In [ ]:
resultados = {}
for idioma in IDIOMAS:
    df = resenas[idioma].reset_index(drop=True)
    puntos = np.array([puntuar(t, LEXICO[idioma], idioma) for t in df["texto"]])
    a_lex, a_lr = [], []
    for semilla in range(20):
        idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.3, stratify=df["etiqueta"], random_state=semilla)
        y = df["etiqueta"].to_numpy()
        umbral, _ = mejor_umbral(puntos[idx_tr], y[idx_tr])
        a_lex.append(accuracy_score(y[idx_te], [clasificar(p, umbral) for p in puntos[idx_te]]))
        modelo = modelo_supervisado().fit(df["texto"].iloc[idx_tr], y[idx_tr])
        a_lr.append(accuracy_score(y[idx_te], modelo.predict(df["texto"].iloc[idx_te])))
    resultados[idioma] = (np.array(a_lex), np.array(a_lr))
    lex, lr = resultados[idioma]
    print(f"{idioma}: léxico {lex.mean():.3f} ± {lex.std():.3f} · TF-IDF + LR {lr.mean():.3f} ± {lr.std():.3f} · el modelo supera al léxico en {int((lr > lex).sum())}/20 particiones")

**¿Qué observas?** La media y la desviación típica (± entre particiones) dan una idea de cuánto se mueve la cifra según la partición. Compara la diferencia entre métodos con esa desviación: si es del mismo orden, hay que ser prudentes al concluir, y solo hablamos de **este corpus sintético**, no de reseñas reales.

---
## 7. Experimento guiado: por qué la bolsa de palabras pierde la negación

Una bolsa de palabras **no tiene orden**: «el servicio no es bueno» y «el servicio es bueno» solo se distinguen por una palabra extra, `no`. Primero lo medimos a mano con el **coseno** entre los dos vectores de recuentos: comparten 4 palabras y la segunda añade una.

In [ ]:
# datos-es
PARES_POS = {   # adjetivo positivo: se espera que la negación BAJE el sentimiento
    "spa": [("el servicio es bueno", "el servicio no es bueno"),
            ("la comida es buena", "la comida no es buena"),
            ("el hotel es limpio", "el hotel no es limpio"),
            ("la aplicación es rápida", "la aplicación no es rápida"),
            ("el envío es puntual", "el envío no es puntual"),
            ("la habitación es cómoda", "la habitación no es cómoda")],
    "ita": [("il servizio è buono", "il servizio non è buono"),
            ("il cibo è buono", "il cibo non è buono"),
            ("l'hotel è pulito", "l'hotel non è pulito"),
            ("l'app è veloce", "l'app non è veloce"),
            ("la consegna è puntuale", "la consegna non è puntuale"),
            ("la camera è comoda", "la camera non è comoda")],
}
PARES_NEG = {   # adjetivo negativo: se espera que la negación SUBA el sentimiento
    "spa": [("el servicio es malo", "el servicio no es malo"),
            ("la comida es mala", "la comida no es mala"),
            ("el hotel es sucio", "el hotel no es sucio"),
            ("la aplicación es lenta", "la aplicación no es lenta"),
            ("el envío es caro", "el envío no es caro"),
            ("el altavoz está roto", "el altavoz no está roto")],
    "ita": [("il servizio è cattivo", "il servizio non è cattivo"),
            ("il cibo è cattivo", "il cibo non è cattivo"),
            ("l'hotel è sporco", "l'hotel non è sporco"),
            ("l'app è lenta", "l'app non è lenta"),
            ("la consegna è cara", "la consegna non è cara"),
            ("l'altoparlante è rotto", "l'altoparlante non è rotto")],
}

In [ ]:
def coseno_bolsa(a, b):
    ta, tb = re.findall(r"\w+", normalizar(a)), re.findall(r"\w+", normalizar(b))
    vocab = sorted(set(ta) | set(tb))
    va, vb = np.array([ta.count(w) for w in vocab]), np.array([tb.count(w) for w in vocab])
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

for idioma in IDIOMAS:
    base, negada = PARES_POS[idioma][0]
    cos = coseno_bolsa(base, negada)
    assert abs(cos - 4 / (2 * 5 ** 0.5)) < 1e-12          # 4 palabras comunes; normas 2 y √5
    print(f"{idioma}: «{base}» frente a «{negada}» → coseno = {cos:.3f}")

Los dos vectores tienen un coseno de 0,894: **casi idénticos**, aunque el sentido es el contrario. Ahora comparamos cuatro enfoques (medimos el cambio del **margen** positivo − negativo al añadir `no`) sobre doce pares mínimos (seis con un adjetivo positivo, «X es bueno» / «X no es bueno», y seis con uno negativo, «X es malo» / «X no es malo»), con los modelos entrenados en el **mismo entrenamiento** de siempre: (a) TF-IDF con **unigramas**, (b) con **unigramas y bigramas**, (c) unigramas y bigramas con **marcas de negación** (`NEG_` delante de las palabras que siguen a un negador, dentro de la ventana) y (d) el **léxico con ventana**.

In [ ]:
def marcar_negacion(texto, idioma, ventana=3):
    """Prefija NEG_ a las palabras que siguen a un negador (hasta `ventana` o hasta un signo de puntuación)."""
    tokens = tokenizar(texto)
    salida, restantes = [], 0
    for t in tokens:
        if not t[0].isalnum() or t in CORTES[idioma]:
            restantes = 0
            salida.append(t)
        elif t in NEGADORES[idioma]:
            restantes = ventana
            salida.append(t)
        elif restantes > 0:
            salida.append("NEG_" + t)
            restantes -= 1
        else:
            salida.append(t)
    return " ".join(salida)

def modelo_con_marcas(idioma):
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=False, token_pattern=r"\S+",
                          preprocessor=lambda t: marcar_negacion(t, idioma))
    return make_pipeline(vec, LogisticRegression(max_iter=1000))

In [ ]:
def margen(modelo, frases):
    """Margen positivo − negativo de la función de decisión (mayor = más positivo)."""
    clases = list(modelo.classes_)
    d = modelo.decision_function(frases)
    return d[:, clases.index("positivo")] - d[:, clases.index("negativo")]

filas = []
for idioma in IDIOMAS:
    modelos = {"(a) unigramas": modelo_supervisado((1, 1)),
               "(b) uni + bigramas": modelo_supervisado((1, 2)),
               "(c) con marcas NEG_": modelo_con_marcas(idioma)}
    for nombre, modelo in modelos.items():
        modelo.fit(train[idioma]["texto"], train[idioma]["etiqueta"])
        fila = {"idioma": idioma, "enfoque": nombre}
        for grupo, pares, signo in (("POS", PARES_POS, -1), ("NEG", PARES_NEG, +1)):
            cambio = margen(modelo, [n for _, n in pares[idioma]]) - margen(modelo, [b for b, _ in pares[idioma]])
            fila[f"{grupo}: pares bien orientados"] = int((signo * cambio > 0).sum())
            fila[f"{grupo}: cambio medio"] = round(float(cambio.mean()), 2)
        filas.append(fila)
    lex = LEXICO[idioma]
    fila = {"idioma": idioma, "enfoque": "(d) léxico con ventana"}
    for grupo, pares, signo in (("POS", PARES_POS, -1), ("NEG", PARES_NEG, +1)):
        cambio = np.array([puntuar(n, lex, idioma) - puntuar(b, lex, idioma) for b, n in pares[idioma]])
        assert (signo * cambio > 0).all()          # propiedad del analizador: invierte los doce pares
        fila[f"{grupo}: pares bien orientados"] = int((signo * cambio > 0).sum())
        fila[f"{grupo}: cambio medio"] = round(float(cambio.mean()), 2)
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** El grupo `POS` («no es bueno») es el caso «fácil»: los modelos supervisados lo orientan bien **porque en este corpus `no` aparece sobre todo en reseñas negativas y ha aprendido un peso negativo**, no porque «entiendan» la negación. El grupo `NEG` («no es malo») lo desenmascara: añadir `no` **sigue bajando** el margen en los tres enfoques supervisados (0 de 6 bien orientados) y el cambio medio es casi el mismo que en `POS`: el efecto de `no` es una constante, independiente de la palabra que niega. Las marcas `NEG_` no lo arreglan aquí porque en un corpus tan pequeño casi no hay ejemplos de `NEG_malo`. El léxico con ventana **invierte por construcción** (lo comprueba el `assert`): es una regla, no un aprendizaje, y tiene sus propios fallos (sección 3).

In [ ]:
filas = []
for idioma in IDIOMAS:
    y = test[idioma]["etiqueta"]
    for nombre, modelo in (("(a)", modelo_supervisado((1, 1))), ("(b)", modelo_supervisado((1, 2))), ("(c)", modelo_con_marcas(idioma))):
        modelo.fit(train[idioma]["texto"], train[idioma]["etiqueta"])
        pred = modelo.predict(test[idioma]["texto"])
        filas.append({"idioma": idioma, "enfoque": nombre, "exactitud en prueba": round(accuracy_score(y, pred), 3)})
print(pd.DataFrame(filas).to_string(index=False))

De (a) a (c) la exactitud sube unos puntos (≈ 5 reseñas de 90 en cada idioma), en la dirección esperada, pero con 90 reseñas de prueba **está dentro del ruido** de la sección 6, y la negación afecta a pocas reseñas (en la sección 8 verás que solo ≈ 8 % de las reseñas en español y ≈ 3 % en italiano tienen una inversión aplicada por el analizador). Es un ejemplo de por qué conviene mirar **casos concretos** además del promedio.

---
## 8. Análisis de errores por categorías

Un único porcentaje esconde **dónde** falla cada método. Construimos una tabla con las predicciones de las 300 reseñas de cada idioma: el léxico (umbral fijado con el entrenamiento) y el modelo supervisado, este último con **validación cruzada de 5 particiones** para que ninguna predicción venga de un modelo que vio esa reseña. Definimos las categorías **mirando solo el texto**, tras inspeccionar errores en la sección anterior y en las primeras pasadas:

- `sin_léxico`: ninguna palabra del texto está en el léxico;
- `con_negación`: alguna palabra quedó invertida por una negación;
- `mixto`: hay palabras positivas y negativas;
- `polaridad_clara`: solo palabras de un signo, sin negación.

Además usamos la columna `dificil` y la **clase real**.

In [ ]:
CAT_SIN = "sin_léxico"      # ninguna palabra del texto está en el léxico
CAT_NEG = "con_negación"    # alguna palabra del léxico quedó invertida por una negación
CAT_MIX = "mixto"                   # hay palabras positivas y negativas
CAT_CLARA = "polaridad_clara"   # solo palabras de un signo, sin negación
ORDEN_CAT = [CAT_SIN, CAT_NEG, CAT_MIX, CAT_CLARA]

def categoria(texto, lexico, idioma):
    """Categoría del texto según lo que ve el léxico (solo mira el texto, no la etiqueta real)."""
    _, aportes = analizar(texto, lexico, idioma)
    if not aportes:
        return CAT_SIN
    if any(base * final < 0 for _, base, final in aportes):
        return CAT_NEG
    if len({np.sign(final) for _, _, final in aportes}) > 1:
        return CAT_MIX
    return CAT_CLARA

In [ ]:
def tabla_predicciones(idioma):
    df = resenas[idioma].reset_index(drop=True).copy()
    lex = LEXICO[idioma]
    df["puntuacion"] = [puntuar(t, lex, idioma) for t in df["texto"]]
    df["pred_lex"] = [clasificar(p, UMBRAL[idioma]) for p in df["puntuacion"]]
    validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    df["pred_lr"] = cross_val_predict(modelo_supervisado(), df["texto"], df["etiqueta"], cv=validacion)
    df["categoria"] = [categoria(t, lex, idioma) for t in df["texto"]]
    df["ok_lex"] = df["pred_lex"] == df["etiqueta"]
    df["ok_lr"] = df["pred_lr"] == df["etiqueta"]
    return df

PRED = {idioma: tabla_predicciones(idioma) for idioma in IDIOMAS}
for idioma, df in PRED.items():
    print(f"{idioma}: exactitud sobre las 300 reseñas · léxico {df['ok_lex'].mean():.3f} · TF-IDF + LR (CV5) {df['ok_lr'].mean():.3f}")

In [ ]:
for idioma, df in PRED.items():
    por_cat = df.groupby("categoria").agg(n=("ok_lex", "size"), acierto_lex=("ok_lex", "mean"), acierto_lr=("ok_lr", "mean")).round(3)
    por_cat = por_cat.reindex([c for c in ORDEN_CAT if c in por_cat.index])
    por_dif = df.groupby("dificil").agg(n=("ok_lex", "size"), acierto_lex=("ok_lex", "mean"), acierto_lr=("ok_lr", "mean")).round(3)
    por_clase = df.groupby("etiqueta").agg(n=("ok_lex", "size"), acierto_lex=("ok_lex", "mean"), acierto_lr=("ok_lr", "mean")).round(3)
    print(f"=== {idioma}: por categoría del texto ===");  print(por_cat)
    print(f"--- {idioma}: por la columna dificil ---");  print(por_dif)
    print(f"--- {idioma}: por clase real ---");  print(por_clase)
    print()

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for eje, idioma in zip(ejes, IDIOMAS):
    df = PRED[idioma]
    tab = df.groupby("categoria")[["ok_lex", "ok_lr"]].mean().reindex([c for c in ORDEN_CAT if c in set(df["categoria"])])
    x = np.arange(len(tab))
    eje.bar(x - 0.2, 1 - tab["ok_lex"], width=0.4, label="léxico", color="#8fb4d9")
    eje.bar(x + 0.2, 1 - tab["ok_lr"], width=0.4, label="TF-IDF + LR", color="#e9a15b")
    eje.set_xticks(x)
    eje.set_xticklabels([f"{c}\n(n={int((df['categoria'] == c).sum())})" for c in tab.index], fontsize=7)
    eje.set_title(idioma)
ejes[0].set_ylabel("tasa de error")
ejes[0].legend()
plt.suptitle("Error por categoría del texto (300 reseñas)")
plt.tight_layout()
plt.show()

**¿Qué observas?** Mira dónde se concentra el error de cada método. Cuando **el léxico no tiene ninguna palabra** de la reseña (`sin_léxico`), solo puede decir «neutro»: acierta en las neutras y falla en las positivas y negativas que usan palabras que no conoce. Y observa la fila de la **clase real**: el `neutro` es la clase más difícil para el léxico, porque «bonito pero pequeño» suma un +1 y un −1 y el resultado depende de cuánto pesa cada palabra.

### Leer los errores (la parte que no se puede automatizar)

Las tablas dicen *cuánto*; para saber *por qué* hay que **leer**. Imprimimos unos pocos errores del léxico en cada categoría con la puntuación que recibieron. Recuerda: cualquier idea que salga de aquí (añadir palabras al léxico, cambiar un umbral) debe validarse en **datos nuevos**; si retocas el léxico mirando estos errores y vuelves a medir sobre las mismas reseñas, la cifra ya no es honesta.

In [ ]:
for idioma, df in PRED.items():
    errores = df[~df["ok_lex"]].sort_values("id")
    print(f"=== {idioma}: {len(errores)} errores del léxico ===")
    for cat in ORDEN_CAT:
        sub = errores[errores["categoria"] == cat]
        print(f"--- {cat}: {len(sub)} errores")
        for _, f in sub.head(3).iterrows():
            print(f"   [{f['etiqueta']} → {f['pred_lex']}, {f['puntuacion']:+.1f}, dificil={f['dificil']}] {f['texto_limpio'][:120]}")
    print()

---
## 9. El sarcasmo y la ironía: un límite que ni el léxico ni la bolsa de palabras resuelven

En «Menuda maravilla de impresora: en un mes ya se ha atascado siete veces» las palabras son positivas y **la opinión es negativa**. Ningún léxico ni ninguna bolsa de palabras ven el contraste con el mundo que hace irónica la frase. Buscamos los casos en que el léxico predice `positivo` para una reseña realmente `negativo` (candidatos a ironía, pero **no todos lo son**: los leeremos).

In [ ]:
for idioma, df in PRED.items():
    candidatos = df[(df["etiqueta"] == "negativo") & (df["pred_lex"] == "positivo")].sort_values("id")
    print(f"=== {idioma}: {len(candidatos)} candidatos (negativas que el léxico ve como positivas) · dificil=1: {int(candidatos['dificil'].sum())} · el modelo supervisado acierta {int(candidatos['ok_lr'].sum())}/{len(candidatos)} ===")
    for _, f in candidatos.head(8).iterrows():
        print(f"   [dificil={f['dificil']}, {f['puntuacion']:+.1f}] {f['texto_limpio'][:125]}")
    print()

**¿Qué observas?** Lee los candidatos. Algunos son ironía o sarcasmo genuinos («Genial la idea de cobrar el pan a tres euros…»); otros son negativos por otra razón (un elogio suelto dentro de una queja, una palabra del léxico usada en otro sentido). Ni el léxico ni el modelo tienen forma de distinguirlos: los rasgos son palabras, y la ironía está en la **relación entre lo que se dice y lo que se espera**. Detectarla exige contexto, conocimiento del mundo o un modelo mucho mayor, y aun así falla.

---
## 10. Resumen y siguiente paso

- El **léxico de polaridad** es transparente (cada decisión se explica palabra a palabra) y no necesita datos etiquetados, pero su cobertura es limitada y su calidad depende del léxico y del dominio. El nuestro es minúsculo y de juguete.
- La **negación** se trata con una ventana de palabras (con sus fallos: distancia, ámbito) y los **intensificadores** con multiplicadores; ambos son reglas escritas a mano.
- El clasificador **supervisado** aprende del corpus lo que el léxico no tiene, pero **necesita etiquetas** y hereda los límites de la bolsa de palabras (la negación se pierde salvo que se codifique).
- **Compara siempre en la misma partición** y sabiendo cuánto ruido hay en la cifra; con corpus pequeños, una diferencia de unos pocos puntos no es concluyente.
- Un solo porcentaje esconde dónde falla el método: analiza los errores **por categorías** y **léelos**.
- La **ironía** y el sarcasmo quedan fuera del alcance de estos métodos.

**Siguiente:** en la clase 7, *Análisis gramatical y entidades*, pasamos de clasificar textos enteros a analizar la estructura de cada frase con spaCy (POS, dependencias y NER). Antes, haz el notebook de práctica de esta clase.